# 复现 Fig. 1：单个暗物质晕的质量与浓度演化

Fig. 1 跟踪一个今天质量为 $10^{12} M_\odot$ 的晕。主体物理公式仍保存在 `haloconcentration.py`：

- `mass_accretion_history` 计算 $M(z)$；
- `concentration_ludlow16` 计算对应的 $C[M(z),z]$；
- 本 Notebook 只选择 Fig. 1 的参数、组织数据、绘图并导出 CSV。

## 1. 导入模块并定位项目目录

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "haloconcentration.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import haloconcentration as hc

PROJECT_ROOT

## 2. 设置 Fig. 1 参数

横轴是对数红移，因此不能包含 $z=0$。这里沿用原程序，在 $0.1\le z\le12$ 上取 400 个对数均匀点。`OUTPUT_TAG` 保留此前约定的 `(1)` 文件名后缀。

In [ ]:
PRESENT_DAY_MASS_MSUN = 1.0e12
REDSHIFT = np.geomspace(0.1, 12.0, 400)
OUTPUT_TAG = "(1)"

REDSHIFT[[0, 100, 200, 300, 399]]

## 3. 计算质量轨迹与浓度轨迹

先用今天的质量 $M_0$ 得到质量吸积史 $M(z)$，再把每个红移处的实际晕质量传入 Ludlow16 模型。这里计算的是 $C[M(z),z]$，不是固定质量 $C(M_0,z)$。

In [ ]:
mass_track_msun = hc.mass_accretion_history(
    PRESENT_DAY_MASS_MSUN,
    REDSHIFT,
    model="ludlow16",
)
concentration_track = hc.concentration_ludlow16(
    mass_track_msun,
    REDSHIFT,
)

# 三列依次是 z、M(z) 和 C[M(z),z]；这里只显示五个代表点。
anchor_indices = [0, 100, 200, 300, 399]
np.column_stack([
    REDSHIFT[anchor_indices],
    mass_track_msun[anchor_indices],
    concentration_track[anchor_indices],
])

## 4. 使用双纵轴绘制 Fig. 1

左纵轴显示无量纲浓度，右纵轴显示晕质量。质量跨越多个数量级，所以右轴采用对数尺度。

In [ ]:
figure, concentration_axis = plt.subplots(figsize=(7.0, 4.8))
mass_axis = concentration_axis.twinx()

concentration_axis.plot(
    REDSHIFT,
    concentration_track,
    color="tab:blue",
    linestyle="--",
    label=r"$C(z)$",
)
mass_axis.plot(
    REDSHIFT,
    mass_track_msun,
    color="tab:red",
    label=r"$M(z)$",
)

concentration_axis.set_xscale("log")
mass_axis.set_yscale("log")
concentration_axis.set_xlim(0.1, 12.0)
concentration_axis.set_xlabel(r"Redshift $z$")
concentration_axis.set_ylabel(r"Concentration $C$", color="tab:blue")
mass_axis.set_ylabel(r"$M(z)\;[M_\odot]$", color="tab:red")
concentration_axis.tick_params(axis="y", colors="tab:blue")
mass_axis.tick_params(axis="y", colors="tab:red")

lines = concentration_axis.lines + mass_axis.lines
labels = [line.get_label() for line in lines]
concentration_axis.legend(lines, labels, loc="upper right")

figure.tight_layout()
figure_path = PROJECT_ROOT / f"fig1_reproduction{OUTPUT_TAG}.png"
figure.savefig(figure_path, dpi=220)
print(f"图片已保存：{figure_path}")
plt.show()

## 5. 导出 Fig. 1 数据

CSV 的三列分别是红移、晕质量和 Ludlow16 浓度。

In [ ]:
csv_path = PROJECT_ROOT / f"fig1_reproduction{OUTPUT_TAG}.csv"
figure1_table = np.column_stack([
    REDSHIFT,
    mass_track_msun,
    concentration_track,
])

np.savetxt(
    csv_path,
    figure1_table,
    delimiter=",",
    header="z,mass_msun,concentration_ludlow16",
    comments="",
)

print(f"数据已保存：{csv_path}")

## 6. 外推说明

Ludlow16 Appendix C 的公开适用范围约到 $z\le9$，而本文 Fig. 1 绘制到 $z=12$。项目中的 `ludlow_nu0` 在 $z>6$ 使用显式、连续的经验延拓，以避开出版多项式在更高红移处变为负值的问题。这是复现约定，不是 Ludlow16 发表的新公式。